# 03 — Native fusion and learning to rank

Recommended runtime: G4 High-RAM, CPU only. Prerequisites: all six completed candidate channels from Notebooks 01–02. RRF k0=60, 100 per channel, and native budget=500 are fixed before Validation. Only Ranking Training queries may be sampled: first 2,000 globally hash-ordered query IDs in full mode, with the fixed seed recorded. Full Validation is never reduced. LR and LambdaMART share one feature allowlist. Injection status remains audit-only; missing source scores remain missing. The final cell selects using native Validation NDCG@20 and publishes an immutable selection manifest. Smoke selection authorizes diagnostic Test only.

Run only in your own Colab session. This notebook mounts Drive and runs the selected pipeline stage. `MODE="smoke"` is the default; smoke outputs are diagnostics, not full H&M results. Full execution has not been run by the authoring agent. Existing source CSVs and CLIP artifacts are read-only. Metadata/images are static snapshots, not proven historical availability.

## Native Colab dependencies
Run in a fresh runtime. A resolver conflict is an error to investigate; do not remove the PyTorch constraints or reinstall CUDA packages to hide it.

In [8]:
import importlib.metadata
import subprocess
import sys
import tempfile

# Use Colab native Python. Never install the repository's uv environment here.
# Pin any installed PyTorch packages so pip cannot replace Colab CUDA builds.
constraints = []
for package in ("torch", "torchvision", "torchaudio"):
    try:
        constraints.append(f"{package}=={importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        pass
with tempfile.NamedTemporaryFile(mode="w", suffix=".txt") as handle:
    handle.write("\n".join(constraints))
    handle.flush()
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--constraint",
            handle.name,
            *[
                "duckdb==1.5.6",
                "pyarrow>=18,<27",
                "scikit-learn>=1.5,<2",
                "lightgbm>=4.5,<5",
                "joblib>=1.4,<2",
            ],
        ],
        check=True,
    )

## Shared protocol and validation contracts
Small helpers are repeated so this notebook has no repository-module dependency.

In [9]:
import hashlib
import json
import math
from datetime import date, timedelta

import numpy as np
import pyarrow as pa

PROTOCOL_VERSION = "1.0"
ARTIFACT_VERSION = "hm-mvp-1.0.0"
FEATURE_SCHEMA_VERSION = "ranking-1.0"
SEED = 20261031
CUTOFF = date(2020, 7, 28)
CHANNELS = ("popularity", "itemcf", "als", "two_tower", "text", "image")
RRF_K0 = 60
CANDIDATES_PER_CHANNEL = 100
FUSED_CANDIDATE_BUDGET = 500
CANDIDATE_SCHEMA = pa.schema(
    [
        ("query_id", pa.string()),
        ("customer_id", pa.string()),
        ("anchor_date", pa.date32()),
        ("article_id", pa.string()),
        ("channel", pa.string()),
        ("raw_score", pa.float32()),
        ("source_rank", pa.int32()),
        ("retriever_version", pa.string()),
    ]
)


def require(condition, message):
    if not condition:
        raise ValueError(message)


def identity(value):
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), default=str)
    return hashlib.sha256(payload.encode()).hexdigest()


def anchor_schedule():
    starts = [
        ("ranking_train", CUTOFF),
        ("validation", date(2020, 8, 16)),
        ("test", date(2020, 9, 4)),
    ]
    return [
        {
            "split": split,
            "anchor_date": start + timedelta(days=offset),
            "window_end": start + timedelta(days=offset + 7),
        }
        for split, start in starts
        for offset in (0, 3, 6, 9, 12)
    ]


def unique_ranked(items):
    return list(dict.fromkeys(items))


def query_metrics(predictions, truth, k=20):
    truth = set(truth)
    require(bool(truth), "Empty ground truth is not an eligible metric denominator.")
    predictions = unique_ranked(predictions)[:k]
    hits = [int(item in truth) for item in predictions]
    recall = sum(hits) / len(truth)
    dcg = sum(hit / math.log2(j + 2) for j, hit in enumerate(hits))
    ideal = sum(1 / math.log2(j + 2) for j in range(min(k, len(truth))))
    cumulative, precision_sum = 0, 0.0
    for j, hit in enumerate(hits, 1):
        cumulative += hit
        precision_sum += hit * cumulative / j
    return {
        "recall": recall,
        "ndcg": dcg / ideal,
        "ap": precision_sum / min(k, len(truth)),
    }


def stable_top(items, scores, limit=100):
    best = {}
    for item, score in zip(items, scores, strict=True):
        score = float(score)
        require(math.isfinite(score), "Nonfinite retrieval score.")
        best[item] = max(best.get(item, -math.inf), score)
    return sorted(best.items(), key=lambda pair: (-pair[1], pair[0]))[:limit]


def fuse_query(rows, excluded=None, budget=500):
    by_item = {}
    keys = set()
    for row in rows:
        if row["channel"] == excluded:
            continue
        key = row["channel"], row["article_id"]
        require(key not in keys, "Duplicate source candidate before fusion.")
        keys.add(key)
        entry = by_item.setdefault(
            row["article_id"],
            {"article_id": row["article_id"], "rrf": 0.0, "sources": {}},
        )
        entry["sources"][row["channel"]] = (row["source_rank"], row["raw_score"])
        entry["rrf"] += 1.0 / (RRF_K0 + row["source_rank"])
    ordered = sorted(by_item.values(), key=lambda row: (-row["rrf"], row["article_id"]))
    return ordered[:budget], len(ordered)


def validate_candidates(table, roster, catalog, channel):
    require(table.schema.equals(CANDIDATE_SCHEMA), "Candidate schema mismatch.")
    query_lookup = {r["query_id"]: r for r in roster}
    seen, groups = set(), {}
    for row in table.to_pylist():
        qid, item = row["query_id"], row["article_id"]
        require(qid in query_lookup, "Candidate references an unknown query.")
        q = query_lookup[qid]
        require(
            row["customer_id"] == q["customer_id"]
            and row["anchor_date"] == q["anchor_date"],
            "Query identity mismatch.",
        )
        require(item in catalog and isinstance(item, str), "Ineligible candidate ID.")
        require(row["channel"] == channel, "Wrong channel in candidate shard.")
        require(
            row["retriever_version"] == ARTIFACT_VERSION, "Retriever version mismatch."
        )
        require((qid, item) not in seen, "Duplicate query/article candidate.")
        require(math.isfinite(row["raw_score"]), "Nonfinite candidate score.")
        seen.add((qid, item))
        groups.setdefault(qid, []).append(row)
    for rows in groups.values():
        rows.sort(key=lambda r: r["source_rank"])
        require(len(rows) <= 100, "More than 100 candidates for a channel.")
        require(
            [r["source_rank"] for r in rows] == list(range(1, len(rows) + 1)),
            "Source ranks are not contiguous from one.",
        )
        expected = stable_top(
            [r["article_id"] for r in rows], [r["raw_score"] for r in rows]
        )
        require(
            [r["article_id"] for r in rows] == [i for i, _ in expected],
            "Unstable score/article ordering.",
        )
    require(
        set(groups) == set(query_lookup), "A query silently disappeared from a channel."
    )


def validate_mapping(ids, rows):
    require(len(ids) == len(rows), "Mapping length mismatch.")
    require(len(ids) == len(set(ids)), "Duplicate article ID in mapping.")
    require(list(rows) == list(range(len(ids))), "Noncontiguous embedding rows.")
    require(
        all(isinstance(x, str) and len(x) == 10 and x.isdigit() for x in ids),
        "Article IDs must preserve ten-digit strings.",
    )


def validate_manifest(manifest, mode, catalog_id=None):
    fields = {
        "artifact_version",
        "protocol_version",
        "mode",
        "input_paths",
        "input_fingerprints",
        "model_parameters",
        "random_seed",
        "row_count",
        "snapshot_or_anchor",
        "catalog_id",
        "feature_schema_version",
        "output_paths",
        "validation_status",
        "files",
        "stage",
    }
    require(fields <= manifest.keys(), "Incomplete artifact manifest.")
    from pathlib import PurePosixPath

    require(
        set(manifest["files"]) == set(manifest["output_paths"]),
        "Output manifest mismatch.",
    )
    for name in manifest["files"]:
        path = PurePosixPath(name)
        require(
            not path.is_absolute() and ".." not in path.parts and "\\" not in name,
            "Unsafe artifact-relative path.",
        )
    require(
        manifest["artifact_version"] == ARTIFACT_VERSION
        and manifest["protocol_version"] == PROTOCOL_VERSION,
        "Artifact/protocol mismatch.",
    )
    require(manifest["mode"] == mode, "Smoke/full artifact mixing is forbidden.")
    require(manifest["validation_status"] == "validated", "Artifact is not complete.")
    if catalog_id is not None:
        require(manifest["catalog_id"] == catalog_id, "Catalog identity mismatch.")


def metric_smoke_checks():
    result = query_metrics(["a", "x", "b"], {"a", "b"}, 3)
    require(result["recall"] == 1, "Recall formula failed.")
    require(abs(result["ap"] - (1 + 2 / 3) / 2) < 1e-12, "AP formula failed.")
    require(query_metrics([], {"a"})["ndcg"] == 0, "Missing recommendations lost.")
    require(
        query_metrics(["a", "a"], {"a", "b"}, 2)["recall"] == 0.5,
        "Duplicate recommendation received extra credit.",
    )
    schedule = anchor_schedule()
    require(len(schedule) == 15, "Anchor count mismatch.")
    require(schedule[-1]["window_end"] == date(2020, 9, 23), "Test overflow.")

## Runtime, storage and one execution mode
Change only `MODE` between `smoke` and `full`. Outputs use `<stage>/mvp_v1/<mode>/` beneath the specified Drive data root. Completion manifests are written last. Changed inputs or parameters require a new artifact version; completed runs are immutable. Local disk checks include a reserve, but cannot guarantee Colab or Drive quota.

In [10]:
import os
import shutil
import time
import uuid
from pathlib import Path

import duckdb
import pyarrow.parquet as pq

DRIVE_ROOT = Path("/content/drive/MyDrive/projects/e_commerce_recsys/data")
LOCAL_ROOT = Path("/content/data/hm")
WORK_ROOT = Path("/content/recsys_work")
MODE = "full"  # 'smoke' or 'full'; smoke results are diagnostics only.
SMOKE_CUSTOMERS = 2000
SMOKE_QUERIES_PER_ANCHOR = 32
MAX_HISTORY_ITEMS = 100  # Most recent distinct items; query representation only.
RUN_NAMESPACE = "mvp_v1"


def resource_policy(mode, cpus, usable_bytes):
    require(mode in ("smoke", "full"), "MODE must be smoke or full.")
    gib = usable_bytes / 1024**3
    workers = max(1, min(int(cpus), 2 if mode == "smoke" else 16))
    memory = max(0.25, min(4 if mode == "smoke" else 32, gib / 4))
    batch = 256
    if mode == "full":
        batch = 4096 if gib >= 32 else 2048 if gib >= 16 else 512 if gib >= 8 else 256
    return {
        "threads": workers,
        "memory_limit": f"{memory:.2f}GB",
        "query_batch_size": batch,
        "target_shard_bytes": (8 if mode == "smoke" else 192) * 1024**2,
        "als_cache_bytes": int(min(1024**3, usable_bytes * 0.05)),
        "sparse_workspace_bytes": int(min(2 * 1024**3, usable_bytes * 0.04)),
        "usable_ram_bytes": int(usable_bytes),
        "available_cpus": int(cpus),
    }


def detect_resources(proc=Path("/proc"), cgroup=Path("/sys/fs/cgroup")):
    cpu_limits = [os.cpu_count() or 1]
    if hasattr(os, "sched_getaffinity"):
        cpu_limits.append(len(os.sched_getaffinity(0)))
    memory_limits = []
    meminfo = proc / "meminfo"
    if meminfo.is_file():
        values = {
            line.split(":")[0]: int(line.split()[1]) * 1024
            for line in meminfo.read_text().splitlines()
            if line.startswith(("MemAvailable:", "MemTotal:"))
        }
        memory_limits.append(
            values.get("MemAvailable", values.get("MemTotal", 4 * 1024**3))
        )
    directories = {cgroup, cgroup / "memory", cgroup / "cpu", cgroup / "cpu,cpuacct"}
    membership = proc / "self/cgroup"
    if membership.is_file():
        for line in membership.read_text().splitlines():
            _, controllers, relative = line.split(":", 2)
            roots = [cgroup] if not controllers else [cgroup / controllers]
            roots += [cgroup / name for name in controllers.split(",") if name]
            for root in roots:
                child = root / relative.lstrip("/")
                # Include ancestor quotas; a child may say 'max' under a finite parent.
                while child == root or root in child.parents:
                    directories.add(child)
                    if child == root:
                        break
                    child = child.parent
    for directory in directories:
        for limit_name, used_name in [
            ("memory.max", "memory.current"),
            ("memory.limit_in_bytes", "memory.usage_in_bytes"),
        ]:
            limit_path, used_path = directory / limit_name, directory / used_name
            if limit_path.is_file():
                text = limit_path.read_text().strip()
                if text.isdigit() and int(text) < 1 << 60:
                    used = int(used_path.read_text()) if used_path.is_file() else 0
                    memory_limits.append(max(0, int(text) - used))
        quota = directory / "cpu.max"
        if quota.is_file():
            amount, period = quota.read_text().split()
            if amount != "max":
                cpu_limits.append(max(1, int(amount) // int(period)))
        quota, period = directory / "cpu.cfs_quota_us", directory / "cpu.cfs_period_us"
        if quota.is_file() and period.is_file() and int(quota.read_text()) > 0:
            cpu_limits.append(max(1, int(quota.read_text()) // int(period.read_text())))
    # Unknown memory is deliberately conservative; no high-RAM assumption on other OSes.
    return min(cpu_limits), min(memory_limits) if memory_limits else 4 * 1024**3


def configure_resources():
    global THREADS, MEMORY_LIMIT, QUERY_BATCH_SIZE, RANK_RESOURCES
    cpus, memory = detect_resources()
    RANK_RESOURCES = resource_policy(MODE, cpus, memory)
    THREADS = RANK_RESOURCES["threads"]
    MEMORY_LIMIT = RANK_RESOURCES["memory_limit"]
    QUERY_BATCH_SIZE = RANK_RESOURCES["query_batch_size"]
    return RANK_RESOURCES


def rss_gib():
    import resource
    import sys

    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return peak / (1024**3 if sys.platform == "darwin" else 1024**2)


def performance_event(operation, started, **fields):
    record = {
        "operation": operation,
        "seconds": time.perf_counter() - started,
        "process_peak_RSS_GiB": rss_gib(),
        **fields,
    }
    RANK_PERFORMANCE.append(record)
    print(json.dumps(record, sort_keys=True))
    return record


RANK_PERFORMANCE = []
RANK_RESOURCES = resource_policy(MODE, *detect_resources())
THREADS = RANK_RESOURCES["threads"]
MEMORY_LIMIT = RANK_RESOURCES["memory_limit"]
QUERY_BATCH_SIZE = RANK_RESOURCES["query_batch_size"]


def sql_string(value):
    return "'" + str(value).replace("'", "''") + "'"


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024**2), b""):
            h.update(chunk)
    return h.hexdigest()


def disk_guard(path, needed_bytes, operation):
    root = Path(path)
    while not root.exists():
        root = root.parent
    free = shutil.disk_usage(root).free
    require(
        free >= needed_bytes + 2 * 1024**3,
        f"{operation}: need {needed_bytes / 1024**3:.2f} GiB plus 2 GiB reserve; "
        f"only {free / 1024**3:.2f} GiB free at {root}. No population truncated.",
    )


def copy_verified(source, target, expected=None):
    source, target = Path(source), Path(target)
    require(source.is_file(), f"Missing prerequisite file: {source}")
    digest = expected or sha256_file(source)
    if target.is_file() and target.stat().st_size == source.stat().st_size:
        if sha256_file(target) == digest:
            return target
    disk_guard(target.parent, source.stat().st_size, "Stage input")
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_suffix(target.suffix + ".partial")
    shutil.copyfile(source, partial)
    require(sha256_file(partial) == digest, "Staged file checksum mismatch.")
    os.replace(partial, target)
    return target


def stage_path(stage, root):
    owned = stage.startswith(
        ("features/", "rankers/", "evaluation/", "candidates/fused/")
    )
    base = Path(root) / "ranking_v2" if owned else Path(root)
    return base / stage / RUN_NAMESPACE / MODE


def load_stage(stage, catalog_id=None, wanted=None):
    remote = stage_path(stage, DRIVE_ROOT)
    marker = remote / "_COMPLETE.json"
    require(marker.is_file(), f"Complete prerequisite missing: {marker}")
    manifest = json.loads(marker.read_text())
    validate_manifest(manifest, MODE, catalog_id)
    require(manifest["stage"] == stage, "Wrong stage manifest.")
    local = stage_path(stage, WORK_ROOT)
    if local.exists():
        unexpected = [
            p
            for p in local.rglob("*")
            if p.is_file()
            and p.relative_to(local).as_posix() not in manifest["files"]
            and p.name != "_COMPLETE.json"
        ]
        if unexpected:
            local.rename(local.with_name(f"{MODE}.incomplete-{time.time_ns()}"))
    local.mkdir(parents=True, exist_ok=True)
    names = manifest["files"] if wanted is None else wanted
    for name in names:
        require(name in manifest["files"], f"Artifact file not declared: {name}")
        info = manifest["files"][name]
        require(
            (remote / name).stat().st_size == info["bytes"], "Published size mismatch."
        )
        require(
            sha256_file(remote / name) == info["sha256"],
            "Published source checksum mismatch.",
        )
        copy_verified(remote / name, local / name, info["sha256"])
    return local, manifest


def reusable(stage, parameters, inputs, catalog_id):
    marker = stage_path(stage, DRIVE_ROOT) / "_COMPLETE.json"
    if not marker.exists():
        return None
    path, manifest = load_stage(stage, catalog_id)
    require(
        manifest["model_parameters"] == parameters,
        "Existing parameters differ; version the run.",
    )
    require(
        manifest["input_fingerprints"] == inputs,
        "Existing inputs differ; version the run.",
    )
    return path, manifest


def runtime_versions():
    from importlib.metadata import PackageNotFoundError, version

    versions = {}
    for name in (
        "duckdb",
        "pyarrow",
        "numpy",
        "scipy",
        "implicit",
        "faiss-cpu",
        "torch",
        "transformers",
        "sentence-transformers",
        "scikit-learn",
        "lightgbm",
        "joblib",
    ):
        try:
            versions[name] = version(name)
        except PackageNotFoundError:
            continue
    return versions


def publish_copy(source, target, expected=None):
    source, target = Path(source), Path(target)
    digest = expected or sha256_file(source)
    if target.exists():
        require(
            target.is_file() and sha256_file(target) == digest,
            f"Conflicting incomplete artifact retained at {target}; "
            "manual review required.",
        )
        return target
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_name(target.name + "." + uuid.uuid4().hex + ".partial")
    shutil.copyfile(source, partial)
    require(sha256_file(partial) == digest, "Publication checksum mismatch.")
    require(not target.exists(), "Concurrent publication detected; retry verification.")
    # Never rename or delete an existing remote artifact.
    os.rename(partial, target)
    return target


def publish(stage, local, parameters, inputs, catalog_id, row_count, snapshot=None):
    publish_started = time.perf_counter()
    local = Path(local)
    files = {}
    for path in sorted(local.rglob("*")):
        if (
            path.is_file()
            and not path.name.startswith(".")
            and path.name != "_COMPLETE.json"
            and not path.name.endswith(".partial")
        ):
            name = path.relative_to(local).as_posix()
            files[name] = {"bytes": path.stat().st_size, "sha256": sha256_file(path)}
    require(bool(files), "Refusing to publish an empty stage.")
    manifest = {
        "stage": stage,
        "artifact_version": ARTIFACT_VERSION,
        "protocol_version": PROTOCOL_VERSION,
        "mode": MODE,
        "input_paths": sorted(inputs),
        "input_fingerprints": inputs,
        "model_parameters": parameters,
        "random_seed": SEED,
        "row_count": int(row_count),
        "snapshot_or_anchor": str(snapshot or CUTOFF),
        "catalog_id": catalog_id,
        "feature_schema_version": FEATURE_SCHEMA_VERSION,
        "output_paths": sorted(files),
        "validation_status": "validated",
        "files": files,
        "runtime_versions": runtime_versions(),
        "execution": {
            "implementation": "ranking-execution-2",
            "resources": RANK_RESOURCES,
        },
    }
    remote = stage_path(stage, DRIVE_ROOT)
    existing = remote / "_COMPLETE.json"
    if existing.exists():
        require(
            json.loads(existing.read_text()) == manifest,
            "Refusing to replace a completed stage.",
        )
        return manifest
    disk_guard(
        remote, sum(x["bytes"] for x in files.values()), "Publish validated stage"
    )
    remote.mkdir(parents=True, exist_ok=True)
    for name, info in files.items():
        publish_copy(local / name, remote / name, info["sha256"])
    # Completion marker is published last, after every checksum is verified.
    marker = local / "_COMPLETE.json"
    marker.write_text(json.dumps(manifest, indent=2, sort_keys=True))
    publish_copy(marker, existing)
    performance_event(
        "publication",
        publish_started,
        stage=stage,
        rows=int(row_count),
        files=len(files),
        bytes=sum(v["bytes"] for v in files.values()),
    )
    return manifest


def new_stage(stage):
    path = stage_path(stage, WORK_ROOT)
    if path.exists() and any(path.iterdir()):
        path.rename(path.with_name(f"{MODE}.incomplete-{time.time_ns()}"))
    path.mkdir(parents=True, exist_ok=True)
    # Completion manifests govern reuse of incomplete local runs.
    return path


def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, sort_keys=True, default=str))


def connect_db():
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    db = duckdb.connect()
    db.execute(f"SET memory_limit={sql_string(MEMORY_LIMIT)}")
    db.execute(f"SET threads={THREADS}")
    db.execute(f"SET temp_directory={sql_string(WORK_ROOT / 'spill')}")
    db.execute("SET preserve_insertion_order=false")
    return db


def export_query(db, query, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_suffix(".partial")
    db.execute(
        f"COPY ({query}) TO {sql_string(partial)} "
        "(FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 65536)"
    )
    os.replace(partial, path)


def view_file(db, name, path):
    db.execute(
        f"CREATE OR REPLACE VIEW {name} AS "
        f"SELECT * FROM read_parquet({sql_string(path)})"
    )


def query_batches(db, anchor):
    previous = ""
    while True:
        rows = db.execute(
            (
                "SELECT query_id, customer_id, anchor_date, split FROM "
                "queries WHERE anchor_date=? AND eligible AND query_id>? "
                "ORDER BY query_id LIMIT ?"
            ),
            [anchor, previous, QUERY_BATCH_SIZE],
        ).fetchall()
        if not rows:
            return
        batch = [
            dict(
                zip(
                    ("query_id", "customer_id", "anchor_date", "split"),
                    row,
                    strict=True,
                )
            )
            for row in rows
        ]
        yield batch
        previous = batch[-1]["query_id"]


def attach_queries(db, batch):
    db.register("batch_queries", pa.Table.from_pylist(batch))


def history_for_batch(db, batch, anchor):
    attach_queries(db, batch)
    # Exact historical counts; only the query preference representation is capped.
    rows = db.execute(
        (
            "WITH grouped AS (\n"
            "        SELECT t.customer_id, t.article_id, count(*) AS n, "
            "max(t.t_dat) AS last_date\n"
            "        FROM transactions t JOIN batch_queries b "
            "USING(customer_id)\n"
            "        WHERE t.t_dat < ? GROUP BY "
            "t.customer_id,t.article_id)\n"
            "        SELECT * FROM grouped QUALIFY row_number() OVER\n"
            "        (PARTITION BY customer_id ORDER BY last_date DESC, "
            "article_id) <= ?\n"
            "        ORDER BY customer_id,last_date DESC,article_id"
        ),
        [anchor, MAX_HISTORY_ITEMS],
    ).fetchall()
    result = {}
    for customer, item, count, last in rows:
        weight = float(count) * math.exp(-(anchor - last).days / 30.0)
        result.setdefault(customer, []).append((item, weight))
    return result


def open_context():
    processed, pm = load_stage(
        "processed",
        wanted=[
            "transactions.parquet",
            "articles.parquet",
            "user_id_map.parquet",
            "article_id_map.parquet",
        ],
    )
    protocol, qm = load_stage("protocol", pm["catalog_id"])
    db = connect_db()
    for name in ("transactions", "articles", "user_id_map", "article_id_map"):
        view_file(db, name, processed / f"{name}.parquet")
    for name in ("eligible_catalog", "anchors"):
        view_file(db, name, protocol / f"{name}.parquet")
    view_file(db, "queries", protocol / "queries" / "*" / "queries.parquet")
    view_file(db, "labels", protocol / "labels" / "*" / "labels.parquet")
    catalog = [
        r[0]
        for r in db.execute(
            "SELECT article_id FROM eligible_catalog ORDER BY article_id"
        ).fetchall()
    ]
    return db, catalog, pm, qm


def runtime_start():
    require(MODE in ("smoke", "full"), "MODE must be smoke or full.")
    from google.colab import drive

    drive.mount("/content/drive")
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
    np.random.seed(SEED)
    settings = configure_resources()
    print(
        {
            "mode": MODE,
            "resources": settings,
            "result_scope": "diagnostic" if MODE == "smoke" else "full protocol",
            "local_free_GiB": round(shutil.disk_usage(WORK_ROOT).free / 1024**3, 2),
        }
    )

## Shared candidate paths and contracts

In [11]:
def training_graph(db, catalog):
    from scipy import sparse

    users = [
        r[0]
        for r in db.execute(
            "SELECT customer_id FROM user_id_map ORDER BY user_idx"
        ).fetchall()
    ]
    db.register(
        "model_items",
        pa.table({"article_id": catalog, "model_row": range(len(catalog))}),
    )
    edges = db.execute(
        "SELECT u.user_idx AS u,i.model_row AS i,count(*) AS n\n"
        "        FROM transactions t JOIN user_id_map u "
        "USING(customer_id)\n"
        "        JOIN model_items i USING(article_id) WHERE t.t_dat <"
        " DATE '2020-07-28'\n"
        "        GROUP BY u.user_idx,i.model_row ORDER BY "
        "u.user_idx,i.model_row"
    )
    # Stream typed arrays; a complete sparse graph, never a dense user-item matrix.
    u_chunks, i_chunks, n_chunks = [], [], []
    for batch in edges.to_arrow_reader(65536):
        u_chunks.append(batch.column(0).to_numpy().astype(np.int32))
        i_chunks.append(batch.column(1).to_numpy().astype(np.int32))
        n_chunks.append(batch.column(2).to_numpy().astype(np.float32))
    require(bool(u_chunks), "Empty retrieval-training graph.")
    graph = sparse.csr_matrix(
        (
            np.concatenate(n_chunks),
            (np.concatenate(u_chunks), np.concatenate(i_chunks)),
        ),
        shape=(len(users), len(catalog)),
        dtype=np.float32,
    )
    graph.sort_indices()
    return graph, users


def itemcf_neighbors(graph, top_k=64, block_size=32):
    from scipy import sparse

    binary = graph.copy()
    binary.data[:] = 1.0
    degree = np.diff(binary.indptr).astype(np.float32)
    # Keep every historical edge; downweight high-degree users instead of dropping them.
    weighted = sparse.diags(1 / np.sqrt(np.maximum(degree, 1))) @ binary
    norms = np.sqrt(np.asarray(weighted.power(2).sum(axis=0)).ravel())
    normalized = (weighted @ sparse.diags(1 / np.maximum(norms, 1e-12))).tocsr()
    blocks = []
    n_items = graph.shape[1]
    for start in range(0, n_items, block_size):
        product = (normalized[:, start : start + block_size].T @ normalized).tocsr()
        row_indices, col_indices, values = [], [], []
        for local_row in range(product.shape[0]):
            a, b = product.indptr[local_row : local_row + 2]
            columns, scores = product.indices[a:b], product.data[a:b]
            keep = (columns != start + local_row) & (scores > 0)
            columns, scores = columns[keep], scores[keep]
            order = np.lexsort((columns, -scores))[:top_k]
            row_indices.extend([local_row] * len(order))
            col_indices.extend(columns[order])
            values.extend(scores[order])
        blocks.append(
            sparse.csr_matrix(
                (values, (row_indices, col_indices)),
                shape=(product.shape[0], n_items),
                dtype=np.float32,
            )
        )
    result = sparse.vstack(blocks, format="csr")
    require(
        np.diff(result.indptr).max(initial=0) <= top_k, "Unbounded ItemCF neighbors."
    )
    return result


def flat_index(vectors):
    import faiss

    index = faiss.IndexFlatIP(vectors.shape[1])
    for start in range(0, len(vectors), 4096):
        index.add(np.ascontiguousarray(vectors[start : start + 4096], dtype=np.float32))
    return index


def index_search(index, queries, ids, limit=100):
    queries = np.ascontiguousarray(queries, dtype=np.float32)
    require(len(ids) == index.ntotal, "FAISS row map length mismatch.")
    if not len(ids):
        return [[] for _ in queries]
    scores, positions = index.search(queries, min(limit + 1, len(ids)))
    output = []
    for q, row_scores, row_positions in zip(queries, scores, positions, strict=True):
        valid = row_positions >= 0
        row_scores, row_positions = row_scores[valid], row_positions[valid]
        # Include every boundary tie before lexicographic ID tie-breaking.
        if len(row_scores) > limit and row_scores[limit] == row_scores[limit - 1]:
            threshold = float(
                np.nextafter(np.float32(row_scores[limit - 1]), np.float32(-np.inf))
            )
            _, row_scores, row_positions = index.range_search(q[None, :], threshold)
        output.append(
            stable_top([ids[int(i)] for i in row_positions], row_scores.tolist(), limit)
        )
    return output


def vector_preferences(batch, histories, vectors, id_to_row):
    result = np.zeros((len(batch), vectors.shape[1]), dtype=np.float32)
    for i, query in enumerate(batch):
        history = histories.get(query["customer_id"], [])
        for item, weight in history:
            row = id_to_row.get(item)
            if row is not None:
                result[i] += vectors[row] * weight
    norms = np.linalg.norm(result, axis=1)
    valid = norms > 1e-12
    result[valid] /= norms[valid, None]
    return result, valid


def candidate_stage(split, channel, anchor):
    return f"candidates/{split}/{channel}/{anchor}"


def generate_channel(
    db, catalog, channel, model_manifest, protocol_manifest, retrieve_batch
):
    catalog_set = set(catalog)
    params = {
        "channel": channel,
        "per_channel": 100,
        "query_batch_size": QUERY_BATCH_SIZE,
        "history_item_limit": MAX_HISTORY_ITEMS,
        "history_decay_days": 30,
    }
    for anchor_info in anchor_schedule():
        anchor, split = anchor_info["anchor_date"], anchor_info["split"]
        stage = candidate_stage(split, channel, anchor)
        inputs = {
            "model_manifest": identity(model_manifest),
            "protocol_manifest": identity(protocol_manifest),
        }
        if reusable(stage, params, inputs, identity(catalog)) is not None:
            continue
        local = new_stage(stage)
        count = db.execute(
            "SELECT count(*) FROM queries WHERE eligible AND anchor_date=?", [anchor]
        ).fetchone()[0]
        disk_guard(local, count * 100 * 180, "Candidate anchor output estimate")
        total, fallback_count, seen_queries = 0, 0, 0
        for batch_number, batch in enumerate(query_batches(db, anchor)):
            histories = history_for_batch(db, batch, anchor)
            results, fallbacks = retrieve_batch(batch, histories)
            require(len(results) == len(batch), "Retriever omitted queries.")
            rows = []
            for query, results_for_query in zip(batch, results, strict=True):
                # Sort after float32 conversion, matching persisted score precision.
                ranked = stable_top(
                    [i for i, _ in results_for_query],
                    [float(np.float32(s)) for _, s in results_for_query],
                )
                for rank, (item, score) in enumerate(ranked, 1):
                    rows.append(
                        {
                            **{
                                k: query[k]
                                for k in ("query_id", "customer_id", "anchor_date")
                            },
                            "article_id": item,
                            "channel": channel,
                            "raw_score": score,
                            "source_rank": rank,
                            "retriever_version": ARTIFACT_VERSION,
                        }
                    )
            table = pa.Table.from_pylist(rows, schema=CANDIDATE_SCHEMA)
            validate_candidates(table, batch, catalog_set, channel)
            pq.write_table(
                table, local / f"shard-{batch_number:05d}.parquet", compression="zstd"
            )
            total += len(rows)
            fallback_count += int(fallbacks)
            seen_queries += len(batch)
        require(seen_queries == count, "Full query population was not processed.")
        write_json(
            local / "diagnostics.json",
            {
                "queries": count,
                "candidate_rows": total,
                "fallback_queries": fallback_count,
            },
        )
        publish(stage, local, params, inputs, identity(catalog), total, anchor)
        print(
            {"channel": channel, "anchor": str(anchor), "queries": count, "rows": total}
        )


def content_retriever(vectors, ids, index, fallback):
    id_to_row = {item: i for i, item in enumerate(ids)}

    def retrieve(batch, histories):
        queries, valid = vector_preferences(batch, histories, vectors, id_to_row)
        results = index_search(index, queries, ids)
        for i in np.flatnonzero(~valid):
            results[int(i)] = fallback
        return results, int((~valid).sum())

    return retrieve

## As-of feature generation, train-only fitting and inference
Historical aggregates are built once per anchor from t_dat < anchor. Native candidates and their features are created before consulting labels. Missing positives are added only to Ranking Training and the separate oracle-assisted Validation diagnostic. LambdaMART groups are deterministic and checked. Source labels are unobserved proxy zeros.

In [12]:
FEATURE_COLUMNS = (
    ["rrf"]
    + [f"{c}_{suffix}" for c in CHANNELS for suffix in ("present", "rank", "score")]
    + [
        "user_purchase_lines",
        "user_recency_days",
        "item_purchase_lines",
        "pair_purchase_lines",
    ]
)


FEATURE_META = [
    ("query_id", pa.string()),
    ("customer_id", pa.string()),
    ("anchor_date", pa.date32()),
    ("article_id", pa.string()),
]
FUSION_SCHEMA = pa.schema(
    FEATURE_META + [(name, pa.float32()) for name in FEATURE_COLUMNS[:-4]]
)
PREDICTION_SCHEMA = pa.schema(
    [
        ("query_id", pa.string()),
        ("article_id", pa.string()),
        ("model", pa.string()),
        ("rank", pa.int32()),
        ("score", pa.float64()),
    ]
)
FEATURE_TABLE_SCHEMA = pa.schema(
    FEATURE_META
    + [(name, pa.float32()) for name in FEATURE_COLUMNS]
    + [("label", pa.int8())]
)


def candidate_anchor_view(db, split, anchor, catalog_id):
    paths, fingerprints = [], {}
    for channel in CHANNELS:
        stage = candidate_stage(split, channel, anchor)
        local, manifest = load_stage(stage, catalog_id)
        require(
            manifest["snapshot_or_anchor"] == str(anchor), "Candidate anchor mismatch."
        )
        baseline_stage = candidate_stage("ranking_train", channel, CUTOFF)
        baseline = json.loads(
            (stage_path(baseline_stage, DRIVE_ROOT) / "_COMPLETE.json").read_text()
        )
        validate_manifest(baseline, MODE, catalog_id)
        require(
            manifest["input_fingerprints"] == baseline["input_fingerprints"]
            and manifest["model_parameters"] == baseline["model_parameters"],
            "Retriever model/protocol/configuration changed across anchors.",
        )
        paths.append(str(local / "shard-*.parquet"))
        fingerprints[stage] = identity(manifest)
    literal = "[" + ",".join(sql_string(p) for p in paths) + "]"
    db.execute(
        f"CREATE OR REPLACE VIEW source_candidates AS "
        f"SELECT * FROM read_parquet({literal})"
    )
    return fingerprints


def asof_history(db, anchor, catalog_id, processed_manifest):
    stage = f"features/history/{anchor}"
    params = {"historical_rule": "t_dat < anchor", "anchor": str(anchor)}
    inputs = {"processed_manifest": identity(processed_manifest)}
    reused = reusable(stage, params, inputs, catalog_id)
    if reused is None:
        local = new_stage(stage)
        n = db.execute(
            "SELECT count(*) FROM transactions WHERE t_dat<?", [anchor]
        ).fetchone()[0]
        disk_guard(local, n * 100, "As-of history aggregates and spill")
        boundary = sql_string(anchor)
        export_query(
            db,
            f"""SELECT customer_id,count(*)::BIGINT AS user_purchase_lines,
            date_diff('day',max(t_dat),DATE {boundary})::INTEGER AS user_recency_days
            FROM transactions WHERE t_dat < DATE {boundary} GROUP BY customer_id""",
            local / "users.parquet",
        )
        export_query(
            db,
            f"""SELECT article_id,count(*)::BIGINT AS item_purchase_lines
            FROM transactions WHERE t_dat < DATE {boundary} GROUP BY article_id""",
            local / "items.parquet",
        )
        export_query(
            db,
            f"""SELECT customer_id,article_id,count(*)::BIGINT AS pair_purchase_lines
            FROM transactions WHERE t_dat < DATE {boundary}
            GROUP BY customer_id,article_id""",
            local / "pairs.parquet",
        )
        manifest = publish(stage, local, params, inputs, catalog_id, n, anchor)
    else:
        local, manifest = reused
    for name in ("users", "items", "pairs"):
        view_file(db, "hist_" + name, local / f"{name}.parquet")
    return manifest


def native_batch(db, batch):
    attach_queries(db, batch)
    raw = (
        db.execute(
            "SELECT s.* FROM source_candidates s JOIN batch_queries b "
            "USING(query_id)\n"
            "        ORDER BY s.query_id,s.channel,s.source_rank"
        )
        .to_arrow_table()
        .to_pylist()
    )
    grouped = {q["query_id"]: [] for q in batch}
    for row in raw:
        grouped[row["query_id"]].append(row)
    output, counts = [], []
    for query in batch:
        qid = query["query_id"]
        require(
            {r["channel"] for r in grouped[qid]} == set(CHANNELS),
            "Incomplete six-channel query; no silent fallback lane omission.",
        )
        fused, before = fuse_query(grouped[qid], budget=FUSED_CANDIDATE_BUDGET)
        counts.append(
            {
                "query_id": qid,
                "pre_budget_count": before,
                "post_budget_count": len(fused),
            }
        )
        for item in fused:
            row = {
                **{k: query[k] for k in ("query_id", "customer_id", "anchor_date")},
                "article_id": item["article_id"],
                "rrf": item["rrf"],
            }
            for channel in CHANNELS:
                rank, score = item["sources"].get(channel, (None, None))
                row.update(
                    {
                        f"{channel}_present": int(rank is not None),
                        f"{channel}_rank": rank,
                        f"{channel}_score": score,
                    }
                )
            output.append(row)
    return output, counts


def historical_features(db, rows):
    require(bool(rows), "No candidate rows to featurize.")
    db.register("feature_base", pa.Table.from_pylist(rows, schema=FUSION_SCHEMA))
    table = db.execute(
        "SELECT b.*,coalesce(u.user_purchase_lines,0)::FLOAT AS "
        "user_purchase_lines,\n"
        "        u.user_recency_days::FLOAT AS user_recency_days,\n"
        "        coalesce(i.item_purchase_lines,0)::FLOAT AS "
        "item_purchase_lines,\n"
        "        coalesce(p.pair_purchase_lines,0)::FLOAT AS "
        "pair_purchase_lines\n"
        "        FROM feature_base b LEFT JOIN hist_users u "
        "USING(customer_id)\n"
        "        LEFT JOIN hist_items i USING(article_id)\n"
        "        LEFT JOIN hist_pairs p USING(customer_id,article_id)"
        "\n"
        "        ORDER BY b.query_id,b.article_id"
    ).to_arrow_table()
    require(table.num_rows == len(rows), "Historical join multiplied candidates.")
    return table.to_pylist()


def batch_truth(db, batch):
    attach_queries(db, batch)
    truth = {q["query_id"]: set() for q in batch}
    for qid, item in db.execute(
        "SELECT l.query_id,l.article_id FROM labels l "
        "JOIN batch_queries b USING(query_id)"
    ).fetchall():
        truth[qid].add(item)
    require(all(truth.values()), "An eligible query has no ground truth.")
    return truth


def inject_rows(native_rows, batch, truth, split):
    require(split in ("ranking_train", "validation"), "Test injection is forbidden.")
    by_query = {q["query_id"]: set() for q in batch}
    for row in native_rows:
        by_query[row["query_id"]].add(row["article_id"])
    added = []
    for query in batch:
        for item in sorted(truth[query["query_id"]] - by_query[query["query_id"]]):
            row = {
                **{k: query[k] for k in ("query_id", "customer_id", "anchor_date")},
                "article_id": item,
                "rrf": None,
            }
            for channel in CHANNELS:
                row.update(
                    {
                        f"{channel}_present": 0,
                        f"{channel}_rank": None,
                        f"{channel}_score": None,
                    }
                )
            added.append(row)
    return added


def labeled_features(rows, truth):
    return [
        {**row, "label": int(row["article_id"] in truth[row["query_id"]])}
        for row in rows
    ]


def validate_groups(rows):
    require(
        rows == sorted(rows, key=lambda r: (r["query_id"], r["article_id"])),
        "Ranker rows must be ordered by query and article.",
    )
    groups, previous = [], None
    seen = set()
    for row in rows:
        if row["query_id"] != previous:
            require(
                row["query_id"] not in seen, "Noncontiguous LambdaMART query group."
            )
            seen.add(row["query_id"])
            groups.append(0)
            previous = row["query_id"]
        groups[-1] += 1
    require(sum(groups) == len(rows), "LambdaMART group size mismatch.")
    require(all(0 < n <= 10000 for n in groups), "Unsupported LambdaMART group size.")
    return groups


def load_ranker_models(catalog_id):
    import joblib
    import lightgbm as lgb

    path, manifest = load_stage("rankers/models", catalog_id)
    schema = json.loads((path / "feature_schema.json").read_text())
    require(
        schema == {"version": FEATURE_SCHEMA_VERSION, "columns": FEATURE_COLUMNS},
        "Ranker feature schema mismatch.",
    )
    return {
        "lr": joblib.load(path / "lr.joblib"),
        "lambdamart": lgb.Booster(model_file=str(path / "lambdamart.txt")),
    }, manifest


def score_feature_rows(rows, models):
    x = feature_matrix(rows)
    scores = {
        "lr": models["lr"].predict_proba(x)[:, 1],
        "lambdamart": models["lambdamart"].predict(x),
    }
    by_query = {}
    for i, row in enumerate(rows):
        by_query.setdefault(row["query_id"], []).append(i)
    output = []
    for model_name, values in scores.items():
        require(np.isfinite(values).all(), "Nonfinite ranking predictions.")
        for qid, positions in by_query.items():
            ranked = stable_top(
                [rows[i]["article_id"] for i in positions],
                [values[i] for i in positions],
                20,
            )
            for rank, (item, value) in enumerate(ranked, 1):
                output.append(
                    {
                        "query_id": qid,
                        "article_id": item,
                        "model": model_name,
                        "rank": rank,
                        "score": value,
                    }
                )
    return output


RANKING_LAYOUT = "ranking_v2"


def ranking_resource_budget(usable_bytes, mode):
    # Allow for sklearn float64/transformed copies and LightGBM bins/gradients.
    budget = int(min(16 * 1024**3, usable_bytes * 0.15))
    bytes_per_row = len(FEATURE_COLUMNS) * 4 * 12
    rows = max(1, budget // bytes_per_row)
    queries = (
        min(50000, max(1, rows // 800))
        if mode == "full"
        else SMOKE_QUERIES_PER_ANCHOR * 5
    )
    return {
        "query_limit": queries,
        "row_limit": rows,
        "working_bytes": budget,
        "bytes_per_row_with_copies": bytes_per_row,
        "assumed_rows_per_query": 800,
        "rule": "md5(query_id || seed),query_id; ranking_train only",
        "seed": SEED,
    }


def rrf_expression(excluded=None):
    result = "0.0::DOUBLE"
    # Original source SQL sorts channels lexically. Match left-to-right double sums.
    for channel in sorted(CHANNELS):
        if channel != excluded:
            result = (
                f"({result} + coalesce(1.0::DOUBLE / "
                f"({RRF_K0} + {channel}_rank),0.0::DOUBLE))"
            )
    return result


def prepare_sql_fusion(db):
    started = time.perf_counter()
    columns = ["count(*) AS source_rows"]
    for channel in CHANNELS:
        columns.extend(
            [
                f"max(source_rank) FILTER (WHERE channel='{channel}') "
                f"AS {channel}_rank",
                f"max(raw_score) FILTER (WHERE channel='{channel}') AS {channel}_score",
            ]
        )
    db.execute(
        "CREATE OR REPLACE TEMP TABLE fusion_union AS "
        "SELECT s.query_id,s.article_id,"
        + ",".join(columns)
        + " FROM source_candidates s JOIN rank_queries q USING(query_id) "
        "GROUP BY s.query_id,s.article_id"
    )
    present = "+".join(f"({ch}_rank IS NOT NULL)::INTEGER" for ch in CHANNELS)
    bad = db.execute(
        f"SELECT count(*) FROM fusion_union WHERE source_rows<>({present})"
    ).fetchone()[0]
    require(bad == 0, "Duplicate source candidate before fusion or unknown channel.")
    complete = " AND ".join(f"max(f.{ch}_rank) IS NOT NULL" for ch in CHANNELS)
    bad = db.execute(f"""SELECT count(*) FROM (SELECT q.query_id FROM rank_queries q
        LEFT JOIN fusion_union f USING(query_id) GROUP BY q.query_id
        HAVING NOT ({complete}))""").fetchone()[0]
    require(bad == 0, "Missing six-channel query coverage.")
    create_fused_view(db, "native_fusion")
    performance_event(
        "sql_fusion",
        started,
        union_rows=db.execute("SELECT count(*) FROM fusion_union").fetchone()[0],
    )


def create_fused_view(db, name, excluded=None):
    included = [f"{c}_rank IS NOT NULL" for c in CHANNELS if c != excluded]
    db.execute(f"""CREATE OR REPLACE TEMP TABLE {name} AS
        SELECT *,row_number() OVER (PARTITION BY query_id ORDER BY rrf_double
        DESC,article_id) AS fusion_rank
        FROM (SELECT *,{rrf_expression(excluded)} AS rrf_double FROM fusion_union
              WHERE {" OR ".join(included)})
        QUALIFY fusion_rank <= {FUSED_CANDIDATE_BUDGET}""")


def fusion_projection(alias="f"):
    fields = [
        f"{alias}.query_id",
        "q.customer_id",
        "q.anchor_date",
        f"{alias}.article_id",
        f"{alias}.rrf_double::FLOAT AS rrf",
    ]
    for c in CHANNELS:
        fields += [
            f"({alias}.{c}_rank IS NOT NULL)::FLOAT AS {c}_present",
            f"{alias}.{c}_rank::FLOAT AS {c}_rank",
            f"{alias}.{c}_score::FLOAT AS {c}_score",
        ]
    return ",".join(fields)


def prepare_scoped_history(db, anchor):
    started = time.perf_counter()
    boundary = sql_string(anchor)
    # One customer-scoped aggregation supplies both user and pair features, including
    # eventual injected items; no future labels are consulted to construct history.
    db.execute(f"""CREATE OR REPLACE TEMP TABLE rank_customer_history AS
        SELECT t.customer_id,t.article_id,count(*)::BIGINT AS
        pair_purchase_lines,max(t_dat) AS last_date
        FROM transactions t SEMI JOIN rank_queries q USING(customer_id)
        WHERE t_dat < DATE {boundary} GROUP BY t.customer_id,t.article_id""")
    db.execute(f"""CREATE OR REPLACE TEMP VIEW hist_users AS SELECT customer_id,
        sum(pair_purchase_lines)::BIGINT AS user_purchase_lines,
        date_diff('day',max(last_date),DATE {boundary})::INTEGER AS user_recency_days
        FROM rank_customer_history GROUP BY customer_id""")
    db.execute(
        "CREATE OR REPLACE TEMP VIEW hist_pairs AS SELECT custom"
        "er_id,article_id,pair_purchase_lines FROM rank_customer"
        "_history"
    )
    # Increment only global item counts, never persist a full global User–Item snapshot.
    state = db.execute(
        "SELECT count(*) FROM information_schema.tables WHERE ta"
        "ble_name='rank_history_cursor'"
    ).fetchone()[0]
    previous = (
        db.execute("SELECT anchor FROM rank_history_cursor").fetchone()[0]
        if state
        else None
    )
    if previous is None or anchor < previous:
        db.execute(f"""CREATE OR REPLACE TEMP TABLE rank_item_counts AS SELECT
        article_id,
            count(*)::BIGINT AS item_purchase_lines FROM transactions WHERE
        t_dat < DATE {boundary} GROUP BY article_id""")
    elif anchor > previous:
        db.execute(f"""CREATE OR REPLACE TEMP TABLE rank_item_counts AS
            SELECT article_id,sum(item_purchase_lines)::BIGINT AS
        item_purchase_lines FROM (
            SELECT * FROM rank_item_counts UNION ALL SELECT article_id,count(*)::BIGINT
            FROM transactions WHERE t_dat >= DATE {sql_string(previous)} AND
        t_dat < DATE {boundary}
            GROUP BY article_id) GROUP BY article_id""")
    db.execute(
        "CREATE OR REPLACE TEMP TABLE rank_history_cursor AS "
        f"SELECT DATE {boundary} AS anchor"
    )
    db.execute(
        "CREATE OR REPLACE TEMP VIEW hist_items AS SELECT * FROM rank_item_counts"
    )
    performance_event(
        "scoped_history",
        started,
        anchor=str(anchor),
        customer_item_rows=db.execute(
            "SELECT count(*) FROM rank_customer_history"
        ).fetchone()[0],
    )


def feature_sql(base):
    return f"""SELECT b.*,coalesce(u.user_purchase_lines,0)::FLOAT AS
        user_purchase_lines,
        u.user_recency_days::FLOAT AS user_recency_days,
        coalesce(i.item_purchase_lines,0)::FLOAT AS item_purchase_lines,
        coalesce(p.pair_purchase_lines,0)::FLOAT AS pair_purchase_lines
        FROM {base} b LEFT JOIN hist_users u USING(customer_id)
        LEFT JOIN hist_items i USING(article_id) LEFT JOIN hist_pairs p
        USING(customer_id,article_id)"""


def write_columnar(db, query, local, prefix, schema=None):
    started = time.perf_counter()
    reader = db.execute(query).to_arrow_reader(65536)
    writer = None
    paths, rows = [], 0
    attempt = uuid.uuid4().hex[:12]
    try:
        for batch in reader:
            table = pa.Table.from_batches([batch])
            if schema is not None:
                table = table.cast(schema)
            if writer is None:
                path = Path(local) / f"{prefix}-{attempt}-{len(paths):04d}.parquet"
                writer = pq.ParquetWriter(path, table.schema, compression="zstd")
                paths.append(path)
            writer.write_table(table, row_group_size=65536)
            rows += len(table)
            if path.stat().st_size >= RANK_RESOURCES["target_shard_bytes"]:
                writer.close()
                writer = None
    finally:
        if writer is not None:
            writer.close()
    # Persist a typed empty addition/audit table too, so reader contracts stay simple.
    if not paths:
        table = db.execute(query + " LIMIT 0").to_arrow_table()
        if schema is not None:
            table = table.cast(schema)
        path = Path(local) / f"{prefix}-{attempt}-0000.parquet"
        pq.write_table(table, path, compression="zstd")
        paths.append(path)
    performance_event(
        "columnar_write",
        started,
        prefix=prefix,
        rows=rows,
        files=len(paths),
        bytes=sum(p.stat().st_size for p in paths),
    )
    return rows


def parquet_view(db, name, local, prefix):
    paths = sorted(Path(local).glob(prefix + "-*.parquet"))
    require(bool(paths), f"Missing {prefix} parquet files.")
    literal = "[" + ",".join(sql_string(p) for p in paths) + "]"
    db.execute(
        f"CREATE OR REPLACE TEMP VIEW {name} AS SELECT * FROM read_parquet({literal})"
    )


def training_sample(db, catalog_id, protocol_manifest):
    stage = "features/training_sample"
    inputs = {"protocol_manifest": identity(protocol_manifest)}
    marker = stage_path(stage, DRIVE_ROOT) / "_COMPLETE.json"
    if marker.exists():
        local, manifest = load_stage(stage, catalog_id)
        require(
            manifest["input_fingerprints"] == inputs, "Training sample inputs changed."
        )
        policy = manifest["model_parameters"]
        require(
            policy["seed"] == SEED and policy["layout"] == RANKING_LAYOUT,
            "Training sample policy changed.",
        )
    else:
        policy = {
            **ranking_resource_budget(RANK_RESOURCES["usable_ram_bytes"], MODE),
            "layout": RANKING_LAYOUT,
        }
        local = new_stage(stage)
        export_query(
            db,
            f"""SELECT query_id FROM queries WHERE eligible AND split='ranking_train'
            ORDER BY md5(query_id || '{SEED}'),query_id LIMIT
        {policy["query_limit"]}""",
            local / "queries.parquet",
        )
        count = pq.ParquetFile(local / "queries.parquet").metadata.num_rows
        manifest = publish(stage, local, policy, inputs, catalog_id, count)
    view_file(db, "selected_training_queries", local / "queries.parquet")
    return policy, manifest


def build_ranking_features(db, catalog, processed_manifest, protocol_manifest, split):
    require(split in ("ranking_train", "validation", "test"), "Unknown split.")
    catalog_id = identity(catalog)
    policy, sample_manifest = (
        training_sample(db, catalog_id, protocol_manifest)
        if split == "ranking_train"
        else (None, None)
    )
    manifests = {}
    for info in anchor_schedule():
        if info["split"] != split:
            continue
        started = time.perf_counter()
        anchor = info["anchor_date"]
        inputs = candidate_anchor_view(db, split, anchor, catalog_id)
        inputs.update(
            processed_manifest=identity(processed_manifest),
            protocol_manifest=identity(protocol_manifest),
        )
        if sample_manifest is not None:
            inputs["training_sample"] = identity(sample_manifest)
        params = {
            "layout": RANKING_LAYOUT,
            "features": FEATURE_COLUMNS,
            "rrf_k0": RRF_K0,
            "budget": FUSED_CANDIDATE_BUDGET,
            "history": "strict t_dat < anchor",
            "injection": split != "test",
            "sample_policy": policy,
        }
        stage = f"features/ranking/{split}/{anchor}"
        reused = reusable(stage, params, inputs, catalog_id)
        if reused is not None:
            manifests[str(anchor)] = reused[1]
            continue
        restriction = (
            "AND query_id IN (SELECT query_id FROM selected_training_queries)"
            if policy
            else ""
        )
        db.execute(f"""CREATE OR REPLACE TEMP TABLE rank_queries AS SELECT
        query_id,customer_id,anchor_date
            FROM queries WHERE eligible AND anchor_date=DATE
        {sql_string(anchor)} {restriction}""")
        count = db.execute("SELECT count(*) FROM rank_queries").fetchone()[0]
        require(count > 0, "No selected ranking queries at anchor.")
        local = new_stage(stage)
        disk_guard(local, count * 600 * 160, "Ranking feature output and SQL spill")
        prepare_sql_fusion(db)
        prepare_scoped_history(db, anchor)
        db.execute(
            "CREATE OR REPLACE TEMP TABLE native_features AS "
            + feature_sql(
                f"(SELECT {fusion_projection()} FROM native_fusion f "
                "JOIN rank_queries q USING(query_id))"
            )
        )
        # Only now attach future labels; no labels can enter native features or fusion.
        db.execute("""CREATE OR REPLACE TEMP TABLE anchor_truth AS SELECT
        DISTINCT l.query_id,l.article_id
            FROM labels l JOIN rank_queries q USING(query_id)""")
        require(
            db.execute("SELECT count(DISTINCT query_id) FROM anchor_truth").fetchone()[
                0
            ]
            == count,
            "Eligible truth coverage mismatch.",
        )
        label_join = " LEFT JOIN anchor_truth t USING(query_id,article_id)"
        native_rows = write_columnar(
            db,
            (
                "SELECT n.*,(t.article_id IS NOT NULL)::TINYINT AS label"
                " FROM native_features n"
            )
            + label_join
            + " ORDER BY query_id,article_id",
            local,
            "native",
            FEATURE_TABLE_SCHEMA,
        )
        added_count = 0
        if split != "test":
            empty = ["NULL::FLOAT AS rrf"]
            for channel in CHANNELS:
                empty += [
                    f"0::FLOAT AS {channel}_present",
                    f"NULL::FLOAT AS {channel}_rank",
                    f"NULL::FLOAT AS {channel}_score",
                ]
            db.execute(
                """CREATE OR REPLACE TEMP TABLE added_base AS SELECT
        q.query_id,q.customer_id,q.anchor_date,t.article_id,"""
                + ",".join(empty)
                + """ FROM anchor_truth t JOIN rank_queries q USING(query_id)
                       ANTI JOIN native_features n USING(query_id,article_id)"""
            )
            added_count = write_columnar(
                db,
                "SELECT *,1::TINYINT AS label FROM ("
                + feature_sql("added_base")
                + ") ORDER BY query_id,article_id",
                local,
                "added",
                FEATURE_TABLE_SCHEMA,
            )
            write_columnar(
                db,
                (
                    "SELECT query_id,article_id,true AS was_injected FROM ad"
                    "ded_base ORDER BY query_id,article_id"
                ),
                local,
                "audit",
            )
        write_columnar(
            db,
            """SELECT u.query_id,count(*)::BIGINT AS pre_budget_count,
            least(count(*),500)::BIGINT AS post_budget_count FROM fusion_union
        u GROUP BY query_id ORDER BY query_id""",
            local,
            "counts",
        )
        write_json(
            local / "feature_schema.json",
            {"version": FEATURE_SCHEMA_VERSION, "columns": FEATURE_COLUMNS},
        )
        write_json(
            local / "pool_counts.json",
            {
                "queries": count,
                "native_rows": native_rows,
                "injected_positive_rows": added_count,
            },
        )
        manifests[str(anchor)] = publish(
            stage, local, params, inputs, catalog_id, native_rows, anchor
        )
        performance_event(
            "ranking_feature_anchor",
            started,
            split=split,
            anchor=str(anchor),
            queries=count,
            native_rows=native_rows,
            added_rows=added_count,
        )
    return manifests


def feature_matrix(rows):
    if not isinstance(rows, pa.Table):
        rows = pa.Table.from_pylist(rows)
    return np.column_stack(
        [rows[name].to_numpy(zero_copy_only=False) for name in FEATURE_COLUMNS]
    ).astype(np.float32)


def feature_pool_view(db, path, injected=False):
    parquet_view(db, "pool_native", path, "native")
    if injected:
        parquet_view(db, "pool_added", path, "added")
        db.execute(
            "CREATE OR REPLACE TEMP VIEW feature_pool AS SELECT * FR"
            "OM pool_native UNION ALL SELECT * FROM pool_added"
        )
    else:
        db.execute(
            "CREATE OR REPLACE TEMP VIEW feature_pool AS SELECT * FROM pool_native"
        )


def score_pool(db, models, local, prefix):
    # Each fixed-size Arrow batch may split a query. Rank only after all batch scores
    # are staged in SQL; no query is lost at a row-group or Parquet-shard boundary.
    temporary = Path(local) / f".scores-{uuid.uuid4().hex}.parquet"
    schema = pa.schema(
        [
            ("query_id", pa.string()),
            ("article_id", pa.string()),
            ("model", pa.string()),
            ("score", pa.float64()),
        ]
    )
    started = time.perf_counter()
    with pq.ParquetWriter(temporary, schema, compression="zstd") as writer:
        reader = db.execute("SELECT * FROM feature_pool").to_arrow_reader(65536)
        for batch in reader:
            table = pa.Table.from_batches([batch])
            x = feature_matrix(table)
            scores = {
                "lr": models["lr"].predict_proba(x)[:, 1],
                "lambdamart": models["lambdamart"].predict(x),
            }
            for model, values in scores.items():
                require(np.isfinite(values).all(), "Nonfinite ranking predictions.")
                writer.write_table(
                    pa.table(
                        {
                            "query_id": table["query_id"],
                            "article_id": table["article_id"],
                            "model": pa.repeat(model, len(table)),
                            "score": pa.array(values, type=pa.float64()),
                        },
                        schema=schema,
                    )
                )
    rows = write_columnar(
        db,
        f"""SELECT query_id,article_id,model,
        row_number() OVER (PARTITION BY query_id,model ORDER BY score
        DESC,article_id)::INTEGER AS rank,score
        FROM read_parquet({sql_string(temporary)}) QUALIFY rank <= 20
        ORDER BY query_id,model,rank""",
        local,
        prefix,
        PREDICTION_SCHEMA,
    )
    temporary.unlink()  # Delete only this freshly created local score spool.
    performance_event("score_pool", started, pool=prefix, prediction_rows=rows)
    return rows


def predict_split(db, catalog, models, model_manifest, split, feature_manifests):
    manifests = {}
    for info in anchor_schedule():
        if info["split"] != split:
            continue
        anchor = info["anchor_date"]
        path, fm = load_stage(f"features/ranking/{split}/{anchor}", identity(catalog))
        require(
            identity(fm) == identity(feature_manifests[str(anchor)]),
            "Feature manifest changed.",
        )
        stage = f"evaluation/{split}_predictions/{anchor}"
        params = {
            "layout": RANKING_LAYOUT,
            "models": ["lr", "lambdamart"],
            "top_k": 20,
            "native_only": split == "test",
        }
        inputs = {"rankers": identity(model_manifest), "features": identity(fm)}
        reused = reusable(stage, params, inputs, identity(catalog))
        if reused is not None:
            manifests[str(anchor)] = reused[1]
            continue
        local = new_stage(stage)
        feature_pool_view(db, path)
        total = score_pool(db, models, local, "native")
        if split == "validation":
            feature_pool_view(db, path, injected=True)
            score_pool(db, models, local, "injected")
        manifests[str(anchor)] = publish(
            stage, local, params, inputs, identity(catalog), total, anchor
        )
    return manifests


def train_rankers(catalog, feature_manifests):
    import joblib
    import lightgbm as lgb
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from threadpoolctl import threadpool_limits

    started = time.perf_counter()
    policies = [
        m["model_parameters"]["sample_policy"] for m in feature_manifests.values()
    ]
    require(
        policies and all(p == policies[0] for p in policies), "Mixed sampling policies."
    )
    policy = policies[0]
    params = {
        "layout": RANKING_LAYOUT,
        "feature_columns": FEATURE_COLUMNS,
        "sample_policy": policy,
        "lr": {"C": 1.0, "max_iter": 300, "solver": "lbfgs"},
        "lambdamart": {
            "objective": "lambdarank",
            "n_estimators": 30 if MODE == "smoke" else 150,
            "num_leaves": 15,
            "learning_rate": 0.05,
            "max_bin": 63,
            "random_state": SEED,
            "n_jobs": THREADS,
            "deterministic": True,
            "force_col_wise": True,
            "verbosity": -1,
        },
    }
    inputs = {
        f"training_features/{a}": identity(m) for a, m in feature_manifests.items()
    }
    reused = reusable("rankers/models", params, inputs, identity(catalog))
    if reused is not None:
        return reused
    paths = []
    for anchor in sorted(feature_manifests):
        path, _ = load_stage(
            f"features/ranking/ranking_train/{anchor}", identity(catalog)
        )
        paths += sorted(path.glob("native-*.parquet")) + sorted(
            path.glob("added-*.parquet")
        )
    db = connect_db()
    literal = "[" + ",".join(sql_string(p) for p in paths) + "]"
    db.execute(
        f"CREATE TEMP VIEW training_pool AS SELECT * FROM read_parquet({literal})"
    )
    groups = db.execute(
        "SELECT query_id,count(*) FROM training_pool GROUP BY qu"
        "ery_id ORDER BY query_id"
    ).fetchall()
    count = sum(r[1] for r in groups)
    require(
        0 < count <= policy["row_limit"] and len(groups) <= policy["query_limit"],
        "Training sample exceeds frozen row/query budget.",
    )
    require(
        count * policy["bytes_per_row_with_copies"]
        <= RANK_RESOURCES["usable_ram_bytes"] * 0.25,
        "Frozen training sample cannot safely fit this runtime; use a larger runtime.",
    )
    scratch = WORK_ROOT / RANKING_LAYOUT / "ranker_fit" / MODE / uuid.uuid4().hex
    scratch.mkdir(parents=True)
    disk_guard(
        scratch,
        count * (len(FEATURE_COLUMNS) * 4 + 1),
        "Columnar ranker training arrays",
    )
    x = np.lib.format.open_memmap(
        scratch / "x.npy",
        mode="w+",
        dtype=np.float32,
        shape=(count, len(FEATURE_COLUMNS)),
    )
    y = np.lib.format.open_memmap(
        scratch / "y.npy", mode="w+", dtype=np.int8, shape=(count,)
    )
    offset = 0
    for batch in db.execute(
        "SELECT * FROM training_pool ORDER BY query_id,article_id"
    ).to_arrow_reader(65536):
        table = pa.Table.from_batches([batch])
        x[offset : offset + len(table)] = feature_matrix(table)
        y[offset : offset + len(table)] = table["label"].to_numpy()
        offset += len(table)
    require(
        offset == count and set(np.unique(y).tolist()) == {0, 1},
        "Training matrix/label contract failed.",
    )
    x.flush()
    y.flush()
    lr = make_pipeline(
        SimpleImputer(strategy="constant", fill_value=0, keep_empty_features=True),
        StandardScaler(),
        LogisticRegression(**params["lr"], random_state=SEED),
    )
    with threadpool_limits(limits=THREADS):
        lr.fit(x, y)
    lm = lgb.LGBMRanker(**params["lambdamart"])
    lm.fit(x, y, group=[r[1] for r in groups])
    local = new_stage("rankers/models")
    joblib.dump(lr, local / "lr.joblib")
    lm.booster_.save_model(str(local / "lambdamart.txt"))
    write_json(
        local / "feature_schema.json",
        {"version": FEATURE_SCHEMA_VERSION, "columns": FEATURE_COLUMNS},
    )
    require(
        np.allclose(
            joblib.load(local / "lr.joblib").predict_proba(x[:32]),
            lr.predict_proba(x[:32]),
        ),
        "LR reload mismatch.",
    )
    require(
        np.allclose(
            lgb.Booster(model_file=str(local / "lambdamart.txt")).predict(x[:32]),
            lm.predict(x[:32]),
        ),
        "LambdaMART reload mismatch.",
    )
    write_json(
        local / "training_summary.json",
        {
            "training_rows": count,
            "training_queries": len(groups),
            "positive_rows": int(y.sum()),
            "sample_policy": policy,
            "pool": "native plus positive additions",
            "injection_distribution_shift": True,
            "zero_labels": "unobserved proxy zeros",
        },
    )
    manifest = publish(
        "rankers/models", local, params, inputs, identity(catalog), count
    )
    db.close()
    performance_event(
        "ranker_fit",
        started,
        rows=count,
        queries=len(groups),
        estimated_working_bytes=count * policy["bytes_per_row_with_copies"],
    )
    return local, manifest

## Macro evaluation and frozen Validation selection
All eligible queries remain in end-to-end denominators, including zero-hit queries. Leave-one-channel-out fusion is rebuilt and re-budgeted. Conditional ranking excludes only queries without retrieved positives and reports that count. Test is not evaluated by this notebook.

In [13]:
def sql_metric_totals(db, prediction_relation, truth_relation, models):
    discounts = ",".join(f"({i},{1 / math.log2(i + 1)!r})" for i in range(1, 21))
    ideals = (
        "["
        + ",".join(
            repr(sum(1 / math.log2(j + 2) for j in range(i))) for i in range(1, 21)
        )
        + "]"
    )
    model_values = ",".join(f"({sql_string(m)})" for m in models)
    return db.execute(f"""WITH truth_counts AS (
        SELECT query_id,count(*) AS n FROM {truth_relation} GROUP BY query_id),
        discounted(rank,discount) AS (VALUES {discounts}),
        model_names(model) AS (VALUES {model_values}),
        hits AS (SELECT p.*, (t.article_id IS NOT NULL)::INTEGER AS hit,
          sum((t.article_id IS NOT NULL)::INTEGER) OVER
          (PARTITION BY p.query_id,p.model ORDER BY p.rank ROWS UNBOUNDED
        PRECEDING) AS cumulative
          FROM {prediction_relation} p LEFT JOIN {truth_relation} t
        USING(query_id,article_id)
          WHERE p.rank<=20),
        per_prediction AS (SELECT h.query_id,h.model,
          sum(hit) AS hits20, sum(hit*d.discount ORDER BY h.rank) AS dcg,
          sum(CASE WHEN h.rank<=12 THEN hit*cumulative::DOUBLE/h.rank ELSE 0
        END ORDER BY h.rank) AS ap
          FROM hits h JOIN discounted d USING(rank) GROUP BY h.query_id,h.model),
        per_query AS (SELECT t.query_id,m.model,
          coalesce(s.hits20,0)::DOUBLE/t.n AS recall,
          coalesce(s.dcg,0)/list_extract({ideals},least(t.n,20)) AS ndcg,
          coalesce(s.ap,0)/least(t.n,12) AS map12
          FROM truth_counts t CROSS JOIN model_names m LEFT JOIN per_prediction
        s USING(query_id,model))
        SELECT model,sum(recall ORDER BY query_id),sum(ndcg ORDER BY query_id),
          sum(map12 ORDER BY query_id),count(*) FROM per_query GROUP BY model
        ORDER BY model""").fetchall()


def evaluate_persisted_split(db, catalog, split):
    require(split in ("validation", "test"), "Invalid evaluation split.")
    totals, denominators, excluded = {}, {}, {"lr": 0, "lambdamart": 0}
    fingerprints = {}

    def add(view, model, metric, total, count):
        key = (view, model, metric)
        totals[key] = totals.get(key, 0.0) + float(total)
        denominators[key] = denominators.get(key, 0) + int(count)

    for info in anchor_schedule():
        if info["split"] != split:
            continue
        started = time.perf_counter()
        anchor = info["anchor_date"]
        fingerprints.update(candidate_anchor_view(db, split, anchor, identity(catalog)))
        path, fm = load_stage(f"features/ranking/{split}/{anchor}", identity(catalog))
        predictions, pm = load_stage(
            f"evaluation/{split}_predictions/{anchor}", identity(catalog)
        )
        require(
            pm["input_fingerprints"]["features"] == identity(fm),
            "Prediction feature lineage changed.",
        )
        fingerprints[f"features/{anchor}"] = identity(fm)
        fingerprints[f"predictions/{anchor}"] = identity(pm)
        db.execute(f"""CREATE OR REPLACE TEMP TABLE rank_queries AS SELECT
        query_id,customer_id,anchor_date
            FROM queries WHERE eligible AND anchor_date=DATE {sql_string(anchor)}""")
        n = db.execute("SELECT count(*) FROM rank_queries").fetchone()[0]
        prepare_sql_fusion(db)
        db.execute("""CREATE OR REPLACE TEMP TABLE eval_truth AS SELECT
        DISTINCT l.query_id,l.article_id
            FROM labels l JOIN rank_queries q USING(query_id)""")
        require(
            db.execute("SELECT count(DISTINCT query_id) FROM eval_truth").fetchone()[0]
            == n,
            "Truth denominator mismatch.",
        )
        parquet_view(db, "eval_native", path, "native")
        mismatch = db.execute("""SELECT count(*) FROM (
            (SELECT query_id,article_id FROM native_fusion EXCEPT SELECT
        query_id,article_id FROM eval_native)
            UNION ALL (SELECT query_id,article_id FROM eval_native EXCEPT
        SELECT query_id,article_id FROM native_fusion))""").fetchone()[0]
        require(mismatch == 0, "Persisted native pool differs from rebuilt SQL fusion.")
        parquet_view(db, "eval_predictions", predictions, "native")
        validate_sql_predictions(db, "eval_predictions", "eval_native", n)
        db.execute("""CREATE OR REPLACE TEMP TABLE truth_counts AS
            SELECT query_id,count(*) AS n FROM eval_truth GROUP BY query_id""")
        recall_fields = []
        recall_keys = []
        for channel in CHANNELS:
            for k in (10, 20, 50, 100):
                key = f"{channel}_{k}"
                recall_fields.append(
                    f"sum(coalesce(f.{channel}_rank<={k},false)::INTEGER) AS {key}"
                )
                recall_keys.append((channel, k, key))
        db.execute(
            "CREATE OR REPLACE TEMP TABLE channel_hits AS "
            "SELECT t.query_id,count(*) AS n,"
            + ",".join(recall_fields)
            + " FROM eval_truth t LEFT JOIN fusion_union f USING(query_id,article_id) "
            "GROUP BY t.query_id"
        )
        sums = ",".join(
            f"sum({key}::DOUBLE/n ORDER BY query_id)" for _, _, key in recall_keys
        )
        values = db.execute("SELECT " + sums + " FROM channel_hits").fetchone()
        for (channel, k, _), value in zip(recall_keys, values, strict=True):
            add("retrieval", channel, f"Recall@{k}", value, n)
        recalls = {}
        for channel in (None, *CHANNELS):
            create_fused_view(db, "eval_fused", channel)
            total = db.execute("""SELECT sum(coalesce(h.hits,0)::DOUBLE/t.n
        ORDER BY t.query_id)
                FROM truth_counts t LEFT JOIN (SELECT f.query_id,count(*) AS
        hits FROM eval_fused f
                JOIN eval_truth g USING(query_id,article_id) GROUP BY
        f.query_id) h USING(query_id)""").fetchone()[0]
            recalls[channel] = total
        add("fusion", "all", "UnionRecall@500", recalls[None], n)
        for channel in CHANNELS:
            add(
                "fusion",
                channel,
                "signed_incremental_Recall@500",
                recalls[None] - recalls[channel],
                n,
            )
        before = db.execute("SELECT count(*) FROM fusion_union").fetchone()[0]
        after = db.execute("SELECT count(*) FROM native_fusion").fetchone()[0]
        add("fusion", "all", "pre_budget_count", before, n)
        add("fusion", "all", "post_budget_count", after, n)
        db.execute("""CREATE OR REPLACE TEMP VIEW eval_ranked AS
            SELECT query_id,article_id,model,rank FROM eval_predictions UNION ALL
            SELECT query_id,article_id,'popularity',popularity_rank FROM fusion_union
            WHERE popularity_rank IS NOT NULL UNION ALL
            SELECT query_id,article_id,'rrf',fusion_rank FROM native_fusion""")
        for model, recall, ndcg, ap, count in sql_metric_totals(
            db, "eval_ranked", "eval_truth", ["popularity", "rrf", "lr", "lambdamart"]
        ):
            for metric, value in [
                ("Recall@20", recall),
                ("NDCG@20", ndcg),
                ("MAP@12", ap),
            ]:
                add("native_end_to_end", model, metric, value, count)
        db.execute("""CREATE OR REPLACE TEMP TABLE conditional_truth AS SELECT
        t.* FROM eval_truth t
            SEMI JOIN eval_native n USING(query_id,article_id)""")
        conditional_count = db.execute(
            "SELECT count(DISTINCT query_id) FROM conditional_truth"
        ).fetchone()[0]
        for model in ("lr", "lambdamart"):
            excluded[model] += n - conditional_count
        for model, _, ndcg, _, count in sql_metric_totals(
            db, "eval_predictions", "conditional_truth", ["lr", "lambdamart"]
        ):
            add("conditional_native", model, "NDCG@20", ndcg, count)
        if split == "validation":
            parquet_view(db, "eval_added", path, "added")
            parquet_view(db, "eval_audit", path, "audit")
            mismatch = db.execute("""SELECT count(*) FROM (
                (SELECT query_id,article_id FROM eval_truth ANTI JOIN
        eval_native USING(query_id,article_id)
                 EXCEPT SELECT query_id,article_id FROM eval_added)
                UNION ALL (SELECT query_id,article_id FROM eval_added EXCEPT
                 SELECT query_id,article_id FROM eval_truth ANTI JOIN
        eval_native USING(query_id,article_id)))""").fetchone()[0]
            require(mismatch == 0, "Injected additions differ from missing positives.")
            mismatch = db.execute("""SELECT count(*) FROM (
                (SELECT query_id,article_id FROM eval_audit WHERE was_injected
        IS TRUE EXCEPT SELECT query_id,article_id FROM eval_added)
                UNION ALL (SELECT query_id,article_id FROM eval_added EXCEPT
        SELECT query_id,article_id FROM eval_audit WHERE was_injected IS
        TRUE))""").fetchone()[0]
            require(mismatch == 0, "Injection audit mismatch.")
            db.execute(
                "CREATE OR REPLACE TEMP VIEW diagnostic_pool AS SELECT *"
                " FROM eval_native UNION ALL SELECT * FROM eval_added"
            )
            parquet_view(db, "diagnostic_predictions", predictions, "injected")
            validate_sql_predictions(db, "diagnostic_predictions", "diagnostic_pool", n)
            injected = db.execute("SELECT count(*) FROM eval_added").fetchone()[0]
            for model, _, ndcg, ap, count in sql_metric_totals(
                db, "diagnostic_predictions", "eval_truth", ["lr", "lambdamart"]
            ):
                add(
                    "oracle_assisted_validation_diagnostic",
                    model,
                    "NDCG@20",
                    ndcg,
                    count,
                )
                add("oracle_assisted_validation_diagnostic", model, "MAP@12", ap, count)
                add(
                    "oracle_assisted_validation_diagnostic",
                    model,
                    "injected_positive_count",
                    injected,
                    count,
                )
        else:
            require(
                not any(name.startswith(("added-", "audit-")) for name in fm["files"]),
                "Test injection artifact forbidden.",
            )
        performance_event(
            "independent_sql_evaluation",
            started,
            split=split,
            anchor=str(anchor),
            queries=n,
        )
    rows = []
    for (view, model, metric), total in sorted(totals.items()):
        count = denominators[view, model, metric]
        rows.append(
            {
                "split": split,
                "mode": MODE,
                "view": view,
                "model": model,
                "metric": metric,
                "value": total
                if metric == "injected_positive_count"
                else total / count,
                "contributing_queries": count,
                "excluded_queries": excluded.get(model, 0)
                if view == "conditional_native"
                else 0,
            }
        )
    for model in ("lr", "lambdamart"):
        if ("conditional_native", model, "NDCG@20") not in totals:
            rows.append(
                {
                    "split": split,
                    "mode": MODE,
                    "view": "conditional_native",
                    "model": model,
                    "metric": "NDCG@20",
                    "value": None,
                    "contributing_queries": 0,
                    "excluded_queries": excluded[model],
                }
            )
    expected = db.execute(
        "SELECT count(*) FROM queries WHERE eligible AND split=?", [split]
    ).fetchone()[0]
    require(
        expected > 0
        and all(
            r["contributing_queries"] == expected
            for r in rows
            if r["view"] != "conditional_native"
        ),
        "Evaluation query population changed.",
    )
    return rows, fingerprints


def validate_sql_predictions(db, predictions, pool, n_queries):
    bad = db.execute(
        f"""SELECT count(*) FROM {predictions} p ANTI JOIN {pool} n
        USING(query_id,article_id)"""
    ).fetchone()[0]
    require(bad == 0, "Prediction outside permitted pool.")
    short = db.execute(f"""SELECT count(*) FROM (
        SELECT query_id,model,count(*) AS n FROM {predictions}
        GROUP BY query_id,model) p
        JOIN (SELECT query_id,count(*) AS n FROM {pool}
        GROUP BY query_id) f USING(query_id)
        WHERE p.n<>least(20,f.n)""").fetchone()[0]
    require(short == 0, "Too few unique ranked items.")
    groups = db.execute(f"""SELECT model,count(*),min(ok) FROM (SELECT query_id,model,
        (count(*)<=20 AND count(*)=count(DISTINCT article_id) AND min(rank)=1
        AND max(rank)=count(*)
         AND count(DISTINCT rank)=count(*)
         AND bool_and(score IS NOT NULL AND isfinite(score))) AS ok
        FROM {predictions} GROUP BY query_id,model) GROUP BY model ORDER BY
        model""").fetchall()
    require(
        len(groups) == 2
        and {r[0] for r in groups} == {"lr", "lambdamart"}
        and all(r[1] == n_queries and r[2] for r in groups),
        "Prediction rank/coverage contract failed.",
    )


def frozen_retriever_contracts(catalog_id):
    result = {}
    for channel in CHANNELS:
        stage = candidate_stage("ranking_train", channel, CUTOFF)
        manifest = json.loads(
            (stage_path(stage, DRIVE_ROOT) / "_COMPLETE.json").read_text()
        )
        validate_manifest(manifest, MODE, catalog_id)
        result[channel] = {
            "inputs": manifest["input_fingerprints"],
            "parameters": manifest["model_parameters"],
        }
    return result


def grouped_predictions(rows):
    result = {}
    seen = set()
    for row in sorted(rows, key=lambda r: (r["model"], r["query_id"], r["rank"])):
        key = row["model"], row["query_id"]
        require((key, row["article_id"]) not in seen, "Duplicate prediction item.")
        seen.add((key, row["article_id"]))
        items = result.setdefault(key, [])
        require(row["rank"] == len(items) + 1, "Prediction ranks must be contiguous.")
        items.append(row["article_id"])
    return result


def choose_validation_model(rows):
    order = ["popularity", "rrf", "lr", "lambdamart"]
    primary = {
        r["model"]: r["value"]
        for r in rows
        if r["split"] == "validation"
        and r["view"] == "native_end_to_end"
        and r["metric"] == "NDCG@20"
    }
    require(set(primary) == set(order), "Incomplete native Validation comparison.")
    return max(order, key=lambda model: (primary[model], -order.index(model)))


def save_metric_csv(path, rows):
    import csv

    require(bool(rows), "No metric records to save.")
    with Path(path).open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def run_fusion_ranking():
    db, catalog, processed_manifest, protocol_manifest = open_context()
    training = build_ranking_features(
        db, catalog, processed_manifest, protocol_manifest, "ranking_train"
    )
    validation = build_ranking_features(
        db, catalog, processed_manifest, protocol_manifest, "validation"
    )
    train_rankers(catalog, training)
    models, model_manifest = load_ranker_models(identity(catalog))
    prediction_manifests = predict_split(
        db, catalog, models, model_manifest, "validation", validation
    )
    rows, fingerprints = evaluate_persisted_split(db, catalog, "validation")
    selected = choose_validation_model(rows)
    # Smoke chooses only a diagnostic pipeline. Full Test refuses a smoke manifest.
    selection = {
        "frozen": True,
        "ranking_layout": RANKING_LAYOUT,
        "mode": MODE,
        "protocol_version": PROTOCOL_VERSION,
        "artifact_version": ARTIFACT_VERSION,
        "catalog_id": identity(catalog),
        "feature_schema_version": FEATURE_SCHEMA_VERSION,
        "selected_model": selected,
        "selection_metric": "native_end_to_end_macro_NDCG@20",
        "selection_split": "validation",
        "diagnostic_only": MODE == "smoke",
        "rrf_k0": RRF_K0,
        "per_channel": 100,
        "native_budget": FUSED_CANDIDATE_BUDGET,
        "ranker_manifest": identity(model_manifest),
        "retriever_contracts": frozen_retriever_contracts(identity(catalog)),
        "validation_predictions": {
            a: identity(m) for a, m in prediction_manifests.items()
        },
        "validation_metrics_sha256": identity(rows),
        "test_policy": "native only; no fitting or parameter changes",
    }
    selection_inputs = {**fingerprints, "ranker_manifest": identity(model_manifest)}
    if (
        reusable(
            "rankers/selection",
            {"selection_rule": selection["selection_metric"]},
            selection_inputs,
            identity(catalog),
        )
        is not None
    ):
        db.close()
        print("Verified completed Validation selection reused.")
        return
    local = new_stage("rankers/selection")
    write_json(local / "selection.json", selection)
    save_metric_csv(local / "validation_ranking_metrics.csv", rows)
    inputs = {**fingerprints, "ranker_manifest": identity(model_manifest)}
    publish(
        "rankers/selection",
        local,
        {"selection_rule": selection["selection_metric"]},
        inputs,
        identity(catalog),
        len(rows),
    )
    db.close()
    print(
        {
            "selected_pipeline": selected,
            "mode": MODE,
            "frozen": True,
            "ranking_layout": RANKING_LAYOUT,
            "scope": "diagnostic only" if MODE == "smoke" else "Validation selection",
        }
    )


def validate_frozen_selection(selection, catalog_id, model_manifest):
    require(selection["frozen"] is True, "Final Test requires a frozen selection.")
    require(
        selection.get("ranking_layout") == RANKING_LAYOUT, "Ranking layout mismatch."
    )
    require(
        selection["mode"] == MODE and selection["catalog_id"] == catalog_id,
        "Selection mode/catalog mismatch.",
    )
    require(
        selection["protocol_version"] == PROTOCOL_VERSION
        and selection["feature_schema_version"] == FEATURE_SCHEMA_VERSION,
        "Selection protocol/feature mismatch.",
    )
    require(
        selection["ranker_manifest"] == identity(model_manifest),
        "Frozen ranker changed.",
    )
    require(
        selection["selection_split"] == "validation",
        "Selection was not made on Validation.",
    )
    require(
        selection["retriever_contracts"] == frozen_retriever_contracts(catalog_id),
        "Frozen retrieval models or candidate settings changed.",
    )
    require(
        selection["native_budget"] == FUSED_CANDIDATE_BUDGET
        and selection["rrf_k0"] == RRF_K0
        and selection["per_channel"] == 100,
        "Frozen candidate configuration changed.",
    )
    if MODE == "full":
        require(
            not selection["diagnostic_only"],
            "Smoke selection cannot authorize final Test.",
        )


def evaluate_validation():
    db, catalog, processed_manifest, protocol_manifest = open_context()
    rows, inputs = evaluate_persisted_split(db, catalog, "validation")
    selection_path, selection_manifest = load_stage(
        "rankers/selection", identity(catalog)
    )
    selection = json.loads((selection_path / "selection.json").read_text())
    require(
        identity(rows) == selection["validation_metrics_sha256"],
        "Independent Validation metrics disagree.",
    )
    require(
        choose_validation_model(rows) == selection["selected_model"],
        "Validation selection changed.",
    )
    if (
        reusable(
            "evaluation/validation_report",
            {"split": "validation"},
            inputs,
            identity(catalog),
        )
        is not None
    ):
        db.close()
        print("Independent Validation verified; existing report reused.")
        return
    local = new_stage("evaluation/validation_report")
    save_metric_csv(
        local / "retrieval_metrics.csv", [r for r in rows if r["view"] == "retrieval"]
    )
    save_metric_csv(
        local / "fusion_metrics.csv", [r for r in rows if r["view"] == "fusion"]
    )
    save_metric_csv(
        local / "validation_ranking_metrics.csv",
        [r for r in rows if r["view"] not in ("retrieval", "fusion")],
    )
    write_json(
        local / "evaluation_manifest.json",
        {
            "mode": MODE,
            "protocol_version": PROTOCOL_VERSION,
            "catalog_id": identity(catalog),
            "selection_manifest": identity(selection_manifest),
            "primary": "native_end_to_end_macro_NDCG@20",
            "test_evaluated": False,
        },
    )
    (local / "summary.md").write_text(
        "# Validation evaluation\n\n"
        f"Mode: {MODE}. Selected pipeline: {selection['selected_model']}.\n\n"
        "All retained eligible queries contribute. Smoke is diagnostic only. "
        "Conditional ranking and oracle-assisted diagnostics are separate views. "
        "This benchmark conditions on future purchases; no online uplift is claimed.\n"
    )
    publish(
        "evaluation/validation_report",
        local,
        {"split": "validation"},
        inputs,
        identity(catalog),
        len(rows),
    )
    db.close()
    print("Independent Validation evaluation matches the frozen selection.")


def evaluate_final_test():
    db, catalog, processed_manifest, protocol_manifest = open_context()
    selection_path, selection_manifest = load_stage(
        "rankers/selection", identity(catalog)
    )
    selection = json.loads((selection_path / "selection.json").read_text())
    models, model_manifest = load_ranker_models(identity(catalog))
    validate_frozen_selection(selection, identity(catalog), model_manifest)
    stage = "evaluation/final_test_report"
    inputs = {
        "selection_manifest": identity(selection_manifest),
        "protocol_manifest": identity(protocol_manifest),
    }
    params = {
        "split": "test",
        "selected_pipeline": selection["selected_model"],
        "native_only": True,
    }
    existing = reusable(stage, params, inputs, identity(catalog))
    if existing is not None:
        print(
            "Final Test report already complete; verified and reused "
            "without recomputing or tuning."
        )
        db.close()
        return
    # Inference only: never call train_rankers or any retriever-fitting function here.
    features = build_ranking_features(
        db, catalog, processed_manifest, protocol_manifest, "test"
    )
    predict_split(db, catalog, models, model_manifest, "test", features)
    rows, lineage = evaluate_persisted_split(db, catalog, "test")
    require(
        not any(r["view"].startswith("oracle") for r in rows),
        "Injected Test evaluation forbidden.",
    )
    local = new_stage(stage)
    save_metric_csv(local / "final_test_metrics.csv", rows)
    save_metric_csv(
        local / "retrieval_metrics.csv", [r for r in rows if r["view"] == "retrieval"]
    )
    save_metric_csv(
        local / "fusion_metrics.csv", [r for r in rows if r["view"] == "fusion"]
    )
    write_json(
        local / "evaluation_manifest.json",
        {
            "mode": MODE,
            "protocol_version": PROTOCOL_VERSION,
            "catalog_id": identity(catalog),
            "selected_pipeline": selection["selected_model"],
            "selection_manifest": identity(selection_manifest),
            "test_lineage": lineage,
            "diagnostic_only": MODE == "smoke",
            "no_test_injection": True,
            "no_test_refit": True,
        },
    )
    (local / "summary.md").write_text(
        "# Final Test evaluation\n\n"
        f"Mode: {MODE}. Frozen selected pipeline: {selection['selected_model']}.\n\n"
        "All candidate budgets and model weights were fixed by Validation. "
        "Other pipeline scores are predeclared comparisons, not a new selection. "
        "No Test injection or refitting occurred. Smoke metrics are diagnostics.\n"
    )
    publish(stage, local, params, inputs, identity(catalog), len(rows))
    db.close()
    print("Final Test report published once. Do not tune against these results.")

## Execute this stage
This is the entry point. It requires the documented completed prerequisites. No models or datasets are downloaded during local source validation.

### Full-scale ranking execution and artifact layout

Only `MODE="smoke"` / `MODE="full"` select execution scope. Resource detection follows
Notebook 01: up to 16 workers, 32 GB DuckDB memory on G4 High-RAM, conservative RAM
fractions on smaller runtimes, and local spill storage. Columnar processing uses
65536-row batches and approximately 192 MiB compressed shards in full mode.

All new ranking-owned durable outputs live under `data/ranking_v2/`, retaining their
existing stage names, `mvp_v1` and mode suffixes inside that root. The six retrieval
candidate/model paths and processed/protocol artifacts remain unchanged. Old ranking
outputs remain untouched and are **not** silently mixed with this layout. Both 03
and 04 contain the same versioned readers. Notebook 04 independently rebuilds fusion
and metrics from the immutable six-channel source candidates; it never trains models.

SQL pivots each anchor once. RRF uses lexically ordered, left-associated DOUBLE sums,
matching the previous source ordering, before Top-500 selection and float32 feature
storage. Leave-one-channel-out fusion recomputes the remaining ordered sum and budget;
it does not subtract a rounded score. Query metrics retain their original macro
semantics; SQL aggregation may differ from Python totals at roundoff precision.

Historical features share a customer-scoped pre-anchor aggregation; global item
counts advance by date deltas. No global User–Item snapshots are persisted. Native
features are constructed before labels are attached. Only missing positive additions
and their audit keys are stored alongside native rows. Training and diagnostic pools
read `native UNION ALL added`; Test contains native rows only. No duplicate full
injected or standalone fused feature files are written.

The Full training sample is selected globally by `md5(query_id || seed),query_id`.
Its recorded budget reserves 12 feature-array copies, up to 15% usable RAM / 16 GiB,
with 800 estimated rows per query and a 50000-query cap. Actual row count and current
RAM are checked before fitting. On 176 GiB usable RAM this permits roughly 19000
queries, rather than 2000. Once published, the sample policy and query IDs are frozen
and reused; Validation/Test are never sampled. LR and LambdaMART hyperparameters,
feature allowlist, positive injection rules and Validation-only selection are retained.

Completed compatible stages are checksum-verified and reused. Conflicting partial
model files fail for manual review; unique feature/prediction shards avoid replacing
old partial files. Do not run concurrent publishers into the same stage. Local score
spools are temporary computation files and are excluded from durable manifests.

Run 03 against the completed 00–02 artifacts, then run 04's independent Validation
check; its separate final-Test entry point preserves the frozen-selection safeguards.
Full H&M runtime, peak RAM, disk spill volume and fit convergence remain unverified
until actual Colab execution. No speedup is claimed from synthetic tests.


In [14]:
runtime_start()
metric_smoke_checks()
run_fusion_ranking()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'mode': 'full', 'resources': {'threads': 16, 'memory_limit': '32.00GB', 'query_batch_size': 4096, 'target_shard_bytes': 201326592, 'als_cache_bytes': 1073741824, 'sparse_workspace_bytes': 2147483648, 'usable_ram_bytes': 186396418048, 'available_cpus': 48}, 'result_scope': 'full protocol', 'local_free_GiB': 192.96}
{"bytes": 688723, "files": 1, "operation": "publication", "process_peak_RSS_GiB": 0.6885223388671875, "rows": 19451, "seconds": 1.4937820530003592, "stage": "features/training_sample"}
{"operation": "sql_fusion", "process_peak_RSS_GiB": 1.9550857543945312, "seconds": 0.4909967530002177, "union_rows": 2021814}
{"anchor": "2020-07-28", "customer_item_rows": 193374, "operation": "scoped_history", "process_peak_RSS_GiB": 2.1334304809570312, "seconds": 0.18425587199999427}
{"bytes": 27100805, "files": 1, "operation": "columnar_write", "prefix": "native"

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{"operation": "sql_fusion", "process_peak_RSS_GiB": 13.233394622802734, "seconds": 3.7708019500000773, "union_rows": 30207385}
{"anchor": "2020-08-16", "customer_item_rows": 2964640, "operation": "scoped_history", "process_peak_RSS_GiB": 13.33935546875, "seconds": 0.19372194400011722}


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{"bytes": 402711201, "files": 2, "operation": "columnar_write", "prefix": "native", "process_peak_RSS_GiB": 28.601932525634766, "rows": 28313437, "seconds": 17.593090305999795}
{"bytes": 4774048, "files": 1, "operation": "columnar_write", "prefix": "added", "process_peak_RSS_GiB": 28.601932525634766, "rows": 138123, "seconds": 0.18423235399995974}
{"bytes": 2265332, "files": 1, "operation": "columnar_write", "prefix": "audit", "process_peak_RSS_GiB": 28.601932525634766, "rows": 138123, "seconds": 0.04296027099962885}
{"bytes": 2055846, "files": 1, "operation": "columnar_write", "prefix": "counts", "process_peak_RSS_GiB": 28.601932525634766, "rows": 60316, "seconds": 0.13329513300004692}
{"bytes": 411807047, "files": 7, "operation": "publication", "process_peak_RSS_GiB": 28.601932525634766, "rows": 28313437, "seconds": 0.8616812179998306, "stage": "features/ranking/validation/2020-08-16"}
{"added_rows": 138123, "anchor": "2020-08-16", "native_rows": 28313437, "operation": "ranking_featu

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (94 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{"anchor": "2020-08-28", "operation": "independent_sql_evaluation", "process_peak_RSS_GiB": 35.03667449951172, "queries": 62037, "seconds": 26.376419230000465, "split": "validation"}
{"bytes": 8148, "files": 2, "operation": "publication", "process_peak_RSS_GiB": 35.03667449951172, "rows": 53, "seconds": 0.021075050000035844, "stage": "rankers/selection"}
{'selected_pipeline': 'lambdamart', 'mode': 'full', 'frozen': True, 'ranking_layout': 'ranking_v2', 'scope': 'Validation selection'}
